In [ ]:

# Global imports & style (hidden)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
# Palette (WCAG-minded; avoid red/green pairing)
BLUE, ORNG, PURP, GREY, TXT, GRID, BG = '#1f77b4', '#ff7f0e', '#7b3294', '#B0B0B0', '#222', '#e8e8e8', 'white'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'axes.edgecolor':'#ccc',
    'axes.grid': True, 'grid.color': GRID, 'grid.alpha': 0.3,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
    'font.size': 11, 'legend.frameon': False
})

def ensure_year_on_dates(ax):
    ax.xaxis.set_major_formatter(DateFormatter('%b %d %Y'))
print('✅ Style ready')


# HERO — Artist‑First Pledge
We invest in music that moves people. This dashboard prioritizes artists and audience experience: clarity over clutter, action over vanity, and accessibility for all.

## S0 — Budget Playbook
- Pick a lane first: Growth (new), Sustain (active), or Revive (dormant).
- Budget follows the choice; charts help validate the bet.
- Color is used sparingly to spotlight the decision; grey carries context.

## S1 — Strategy Pros/Cons (A/B/C)
- A. Growth: fast tests, thumbnail/title sprints, release cadence.
- B. Sustain: compound wins, keep fans warm, feature timing.
- C. Revive: catalog moments, collab hooks, re‑edits.

## H2 — Artist Spotlight
Quotes from socials + days ≥55 this year (if data present).

In [ ]:

# Spotlight card — per-artist summary and days ≥55 if available
if 'vids_safe' in globals():
    df = vids_safe.copy()
    df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
    df['pub_date'] = df['published_at'].dt.floor('D')

    # Base stats
    base = df.groupby('artist_name').agg(
        videos=('video_id','nunique'),
        views=('view_count','sum')
    ).sort_values('views', ascending=False)

    # Days ≥55 via momentum_daily if present
    days55 = None
    try:
        if 'momentum_daily' in globals():
            md = momentum_daily.copy()
            md['date'] = pd.to_datetime(md['date']).dt.floor('D')
            hits = md.loc[md['momentum_score']>=55, ['video_id','date']]
            amap = df[['video_id','artist_name']].drop_duplicates()
            days55 = hits.merge(amap, on='video_id', how='left')                         .groupby('artist_name')['date'].nunique().rename('days_55')
    except Exception:
        days55 = None

    spotlight = base
    if days55 is not None:
        spotlight = spotlight.join(days55, how='left').fillna({'days_55':0}).astype({'days_55':'int64'})

    display(spotlight.head(6))
else:
    print('ℹ️ vids_safe not found — run metrics cell')


## Chart 1.1 — Roster Overview (per‑artist key stats)

In [ ]:

if 'vids_safe' in globals():
    df = vids_safe.copy()
    if 'like_rate' not in df.columns:
        df['like_rate'] = (df['like_count'] / df['view_count'].replace(0,np.nan)).fillna(0).clip(0,1)
    tbl = df.groupby('artist_name').agg(
        videos=('video_id','nunique'),
        views=('view_count','sum'),
        median_like_rate=('like_rate','median')
    ).sort_values('views', ascending=False)
    display(tbl.head(15))
else:
    print('ℹ️ vids_safe not found')


## Chart 1.2 — Engagement Patterns by Artist (interactive)

In [ ]:

try:
    import plotly.express as px
    from tools.advanced_charts import TXT
    PLOTLY = True
except Exception:
    PLOTLY = False

if 'vids_safe' in globals():
    df = vids_safe.copy()
    df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
    if 'like_rate' not in df.columns:
        df['like_rate'] = (df['like_count']/df['view_count'].replace(0,np.nan)).fillna(0).clip(0,1)
    df['pub_date'] = df['published_at'].dt.floor('D')
    if PLOTLY:
        top = df.groupby('artist_name').size().sort_values(ascending=False).head(6).index.tolist()
        d = df[df['artist_name'].isin(top)].copy()
        if 'title' not in d.columns: d['title'] = d.get('video_title', d.get('video_id','Unknown'))
        d['hover'] = ('<b>'+d['title'].astype(str).str[:60]+'</b><br>'
                      'Artist: '+d['artist_name'].astype(str)+'<br>'
                      'Like Rate: '+(d['like_rate']*100).round(1).astype(str)+'%<br>'
                      'Views: '+d['view_count'].apply(lambda x: f"{x:,}")+'<br>'
                      'Date: '+d['pub_date'].dt.strftime('%b %d %Y'))
        fig = px.scatter(d, x='pub_date', y='like_rate', color='artist_name', size='view_count',
                         hover_name='title', hover_data={'hover':True,'pub_date':False,'like_rate':False},
                         size_max=18, title='Like‑rate shows quality → hover for titles')
        fig.update_yaxes(tickformat='.0%')
        fig.update_xaxes(tickformat='%b %d %Y')
        fig.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(size=12, color=TXT))
        fig.show()
    else:
        print('Install plotly for interactivity: pip install plotly')
else:
    print('ℹ️ vids_safe not found')


## Chart 1.3 — Performance Trends (Exec view)

In [ ]:

# Minimal exec trend: daily total views
if 'vids_safe' in globals() and 'view_count' in vids_safe.columns:
    d = (vids_safe.assign(published_at=pd.to_datetime(vids_safe['published_at']).dt.floor('D'))
         .groupby('published_at')['view_count'].sum())
    fig, ax = plt.subplots(figsize=(12,4)); ax.plot(d.index, d.values, color=BLUE, lw=2)
    ensure_year_on_dates(ax); ax.set_ylabel('Views'); ax.set_title('Roster view trend (sum of views)')
    ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
else:
    print('ℹ️ views not available')


## Chart 1.3a — Analyst: Base‑100 + MAD outliers

In [ ]:

from tools.advanced_charts import base100, modified_z
if 'vids_safe' in globals() and 'view_count' in vids_safe.columns:
    d = (vids_safe.assign(published_at=pd.to_datetime(vids_safe['published_at']).dt.floor('D'))
         .groupby('published_at')['view_count'].sum())
    idx = base100(d)
    mz = modified_z(idx)
    out = (mz.abs()>3.5) & idx.notna()
    fig, ax = plt.subplots(figsize=(12,4)); ax.plot(idx.index, idx.values, color=BLUE, lw=1.8)
    ax.scatter(idx.index[out], idx[out], color=ORNG, s=18, zorder=3)
    ensure_year_on_dates(ax); ax.set_ylabel('Base‑100 index'); ax.set_title('Base‑100 vs 30‑day median (orange = robust outliers)')
    ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
else:
    print('ℹ️ views not available')


## Chart 2.1 — Threshold Comparison (55 vs 75) + Video‑Day explainer

In [ ]:

THRESHOLDS = {'pre_breakout':55,'legacy':75,'breakout':60} if 'THRESHOLDS' not in globals() else THRESHOLDS
if 'momentum_daily' in globals():
    def _daily_hits(d,t):
        return d.loc[d['momentum_score']>=t].groupby('date')['video_id'].nunique().rename(f'videos_at_{int(t)}')
    d55=_daily_hits(momentum_daily, THRESHOLDS['pre_breakout']); d75=_daily_hits(momentum_daily, THRESHOLDS['legacy'])
    q=(pd.concat([d55,d75],axis=1).fillna(0).reset_index().sort_values('date'))
    q['incremental_videos']=q[f"videos_at_{int(THRESHOLDS['pre_breakout'])}"]-q[f"videos_at_{int(THRESHOLDS['legacy'])}"]
    q['cumulative_incremental']=q['incremental_videos'].cumsum()
    fig,ax=plt.subplots(figsize=(12,4)); ax.plot(pd.to_datetime(q['date']), q['cumulative_incremental'], color=BLUE, lw=2.5)
    ensure_year_on_dates(ax); ax.set_ylabel('Cumulative Additional Video‑Days')
    ax.set_title(f"Threshold {THRESHOLDS['pre_breakout']} adds {int(q['incremental_videos'].sum()):,} video‑days")
    ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
else:
    print('ℹ️ momentum_daily not found')


## Chart 2.2 — Threshold Impact vs Ops Capacity (cumulative video‑days)

In [ ]:

OPS_CAP = globals().get('OPS_CAP_VIDEO_DAYS_PER_WEEK', None)
if 'momentum_daily' in globals():
    def _daily_hits(d,t): return d.loc[d['momentum_score']>=t].groupby('date')['video_id'].nunique()
    d55=_daily_hits(momentum_daily,55)
    q=(d55.sort_index().reindex(pd.date_range(d55.index.min(), d55.index.max(), name='date'), fill_value=0).reset_index())
    q['cum']=q['video_id'].cumsum()
    fig,ax=plt.subplots(figsize=(12,4)); ax.plot(pd.to_datetime(q['date']), q['cum'], color=BLUE, lw=2.2)
    if OPS_CAP:
        per_day = OPS_CAP/7.0; x = pd.to_datetime(q['date']); cap=np.arange(len(x))*per_day
        ax.plot(x, cap, color=GREY, lw=1.5, ls='--')
    ensure_year_on_dates(ax); ax.set_ylabel('Cumulative Video‑Days'); ax.set_title('Threshold impact vs ops capacity (dashed)')
    ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
else:
    print('ℹ️ momentum_daily not found')


## Chart 2.3 — Momentum Bar Race (weekly)

In [ ]:
print('ℹ️ Use tools/momentum_bar_race.py in pipeline to render or export the animation.')

## KPI‑22 — Breakout Duration & Pre‑Warning (Top‑10 by artist; cap‑aware)

In [ ]:

from tools.advanced_charts import compute_kpi22_episodes
if 'momentum_daily' in globals() and 'vids_safe' in globals():
    eps = compute_kpi22_episodes(momentum_daily, pre=55, brk=60, cap_hours=720)
    if not eps.empty:
        m = vids_safe[['video_id','artist_name']].drop_duplicates()
        eps = eps.merge(m, on='video_id', how='left')
        agg = eps.groupby('artist_name').agg(
            mean_dur=('duration_days','mean'),
            mean_warn=('pre_warning_hours','mean'),
            any_capped=('capped','max')
        ).reset_index()
        top_dur = agg.nlargest(10,'mean_dur').sort_values('mean_dur')
        top_warn= agg.nlargest(10,'mean_warn').sort_values('mean_warn')
        fig,(ax1,ax2)=plt.subplots(1,2, figsize=(16,6))
        # Left: breakout duration
        y=np.arange(len(top_dur)); b=ax1.barh(y, top_dur['mean_dur'], color=ORNG, height=0.6)
        ax1.set_yticks(y); ax1.set_yticklabels(top_dur['artist_name']); ax1.set_xlabel('Avg Breakout Duration (days)')
        for i,v in enumerate(top_dur['mean_dur']): ax1.text(v, i, f'{v:.1f}', va='center', ha='left', fontsize=10)
        # Right: pre-warning hours
        y2=np.arange(len(top_warn)); b2=ax2.barh(y2, top_warn['mean_warn'], color=PURP, height=0.6)
        cap_mask=agg.set_index('artist_name').loc[top_warn['artist_name'],'any_capped'].values
        for i,cap in enumerate(cap_mask):
            if cap: b2[i].set_hatch('///')
        ax2.set_yticks(y2); ax2.set_yticklabels(top_warn['artist_name']); ax2.set_xlabel('Avg Pre‑Breakout Warning (hours)')
        for i,v in enumerate(top_warn['mean_warn']): ax2.text(v, i, f'{v:.0f}', va='center', ha='left', fontsize=10)
        ax1.set_title('KPI‑22 (1/2): Breakouts — avg duration', pad=10)
        ax2.set_title('KPI‑22 (2/2): Pre‑warning — avg hours (///=cap)', pad=10)
        for a in (ax1,ax2): a.grid(True, axis='x', alpha=0.3, color=GRID)
        plt.tight_layout(); plt.show()
    else:
        print('⚠️ No breakouts detected (score ≥60).')
else:
    print('ℹ️ momentum_daily / vids_safe not found')


## FYI — Sentiment (diverging 100% bars)

In [ ]:

from tools.advanced_charts import diverging_sentiment_df, plot_diverging_sentiment
if 'comments_safe' in globals() and set(['artist_name','pos','neu','neg']).issubset(comments_safe.columns):
    d = comments_safe.groupby('artist_name')[['pos','neu','neg']].sum().reset_index()
    dd = diverging_sentiment_df(d, 'pos','neu','neg')
    try:
        fig = plot_diverging_sentiment(dd, 'artist_name'); fig.show()
    except RuntimeError:
        print('Install plotly for sentiment viz: pip install plotly')
else:
    print('ℹ️ sentiment columns not available')


## FYI — Publish Hour vs Avg Views/Day

In [ ]:

if 'vids_safe' in globals():
    df=vids_safe.copy()
    df['published_at']=pd.to_datetime(df['published_at'], errors='coerce')
    df['pub_date']=df['published_at'].dt.floor('D')
    now=pd.Timestamp.today().normalize(); df['age_days']=(now-df['pub_date']).dt.days.clip(lower=1)
    df['views_per_day']=(df['view_count']/df['age_days']).replace([np.inf,np.nan],0.0)
    df['hour']=df['published_at'].dt.hour
    perf=df.groupby('hour')['views_per_day'].mean().reindex(range(24),fill_value=0.0)
    fig,ax=plt.subplots(figsize=(12,4)); bars=ax.bar(perf.index, perf.values, color=GREY, edgecolor=TXT, alpha=0.75)
    best=int(perf.idxmax()); bars[best].set_color(BLUE)
    for h,v in perf.items(): ax.text(h,v,f'{v:,.0f}',ha='center',va='bottom',fontsize=9,color=TXT)
    ax.set_xlabel('Publish Hour (24h)'); ax.set_ylabel('Avg Views/Day'); ax.set_title(f'Best hour: {best:02d}:00')
    plt.tight_layout(); plt.show()
else:
    print('ℹ️ vids_safe not found')


## FYI — Comment Length Distribution

In [ ]:

DISPLAY_LIMIT=800
if 'comments_safe' in globals() and 'text' in comments_safe.columns:
    lens=comments_safe['text'].astype(str).str.len(); total=len(lens); out=(lens>DISPLAY_LIMIT).sum()
    lens_display=lens.clip(upper=DISPLAY_LIMIT)
    fig,ax=plt.subplots(figsize=(12,4)); ax.hist(lens_display, bins=40, color=GREY, edgecolor=TXT, alpha=0.85, range=(0,DISPLAY_LIMIT))
    pct=(out/total*100 if total else 0); ax.set_xlabel('Comment length (chars)'); ax.set_ylabel('Count')
    ax.set_title(f'Most comments are short (0–{DISPLAY_LIMIT}); {out:,} long (> {DISPLAY_LIMIT}) = {pct:.1f}%')
    plt.tight_layout(); plt.show()
else:
    print('ℹ️ comments not available')


## Data Provenance
This dashboard uses YouTube Data API v3 metrics gathered by our ETL. If synthetic or subset data is used, labels on charts will indicate it.

## Quality Helpers (appendix)
- Direct labels; legends avoided
- Use grey for context; ≤10% highlight color
- Modified‑Z outliers (|z|>3.5)
- Contrast ≥4.5:1 for text
- Year always visible on date axes